# Auswertung der Studienergebnisse

Dieses Notebook wertet die JSON-Dateien im Ordner `ergebnisse/` aus und erzeugt Plots im Ordner `plots/`.

Fuer jede Ergebnisdatei wird ein Unterordner `plots/<dateiname>/` angelegt. Dateien, fuer die bereits Plots existieren, werden uebersprungen (ausser `FORCE_REGENERATE = True`).

Erzeugte Plots pro Teilnehmer:
- Distanz-normalisierte Zeit pro Block
- Benoetigte Flicks pro Block (nur bei vorhandenen Flick-Daten)
- Entwicklung der getesteten Parameter pro Block

In [16]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Auf True setzen, um alle Plots neu zu erzeugen (auch wenn sie schon existieren)
FORCE_REGENERATE = False


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'ergebnisse').is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root()
ERGEBNISSE_DIR = REPO_ROOT / 'ergebnisse'
PLOTS_DIR = REPO_ROOT / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)

print('Repo root:  ', REPO_ROOT)
print('Ergebnisse: ', ERGEBNISSE_DIR)
print('Plots:      ', PLOTS_DIR)

Repo root:   C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app
Ergebnisse:  C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\ergebnisse
Plots:       C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\plots


In [17]:
def participant_display_name(participant):
    first = (participant.get('firstName') or '').strip()
    last = (participant.get('lastName') or '').strip()
    name = (first + ' ' + last).strip()
    return name or participant.get('participantId', 'unbekannt')


def short_id(participant):
    pid = participant.get('participantId') or 'unbekannt'
    return pid.split('-')[0]


def load_results_file(path):
    with open(path, encoding='utf-8') as fh:
        data = json.load(fh)
    if isinstance(data, dict):
        data = [data]
    return data


def attempts_dataframe(participant):
    rows = []
    for block in participant.get('attemptBlocks', []):
        run_number = block.get('runNumber')
        params = block.get('parameterSet', {})
        for attempt in block.get('attempts', []):
            rows.append({
                'participantId': participant.get('participantId'),
                'participant': participant_display_name(participant),
                'runNumber': run_number,
                'attemptInBlock': attempt.get('attemptInBlock'),
                'targetNumber': attempt.get('targetNumber'),
                'scrollDistance': attempt.get('scrollDistance'),
                'timeMs': attempt.get('timeMs'),
                'flickCount': attempt.get('flickCount'),
                'switchbackCount': attempt.get('switchbackCount'),
                'timestamp': attempt.get('timestamp'),
                'decelerationRate': params.get('decelerationRate'),
                'scrollFriction': params.get('scrollFriction'),
                'inflexion': params.get('inflexion'),
            })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.dropna(subset=['timeMs', 'scrollDistance']).reset_index(drop=True)
        df['msPer1000px'] = df['timeMs'] / df['scrollDistance'].replace(0, np.nan) * 1000
        # flickCount / switchbackCount only exist for newer runs; keep them numeric.
        df['flickCount'] = pd.to_numeric(df['flickCount'], errors='coerce')
        df['switchbackCount'] = pd.to_numeric(df['switchbackCount'], errors='coerce')
    return df


def has_flick_data(df):
    return 'flickCount' in df.columns and not df['flickCount'].dropna().empty


def metrics_dataframe(participant):
    rows = []
    for m in participant.get('parameterBlockMetrics', []):
        model = m.get('model', {})
        completion = m.get('completionTimeMs', {})
        rows.append({
            'blockNumber': m.get('blockNumber'),
            'generatedFromAttemptCount': m.get('generatedFromAttemptCount'),
            'meanCompletionMs': completion.get('mean'),
            'medianCompletionMs': completion.get('median'),
            'bestObservedNormalizedTime': model.get('bestObservedNormalizedTime'),
            'predictedCandidateNormalizedTime': model.get('predictedCandidateNormalizedTime'),
            'predictedCurrentNormalizedTime': model.get('predictedCurrentNormalizedTime'),
            'acquisitionValue': model.get('acquisitionValue'),
            'candidateUncertaintyStd': model.get('candidateUncertaintyStd'),
        })
    return pd.DataFrame(rows)

In [18]:
def plots_subdir_for(path):
    return PLOTS_DIR / path.stem


def has_plots(path):
    out_dir = plots_subdir_for(path)
    return out_dir.is_dir() and any(out_dir.glob('*.png'))


result_files = sorted(ERGEBNISSE_DIR.glob('*.json'))
pending_files = [p for p in result_files if FORCE_REGENERATE or not has_plots(p)]

print('Gefundene Ergebnisdateien:', len(result_files))
for p in result_files:
    status = 'AUSSTEHEND' if p in pending_files else 'hat bereits Plots'
    print('  -', p.name, '=>', status)

Gefundene Ergebnisdateien: 3
  - touch-scrolling-data-2026-09-14.json => hat bereits Plots
  - touch-scrolling-data-p2-Kr-Kr-2026-09-14.json => hat bereits Plots
  - touch-scrolling-data-p3-Ur-Kr-2026-09-14.json => AUSSTEHEND


In [21]:
def save_fig(fig, out_dir, name):
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / name
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return out_path


def plot_normalized_time_per_block(df, label, out_dir, prefix):
    grouped = df.groupby('runNumber')['msPer1000px'].agg(['mean', 'std']).reset_index()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.errorbar(grouped['runNumber'], grouped['mean'], yerr=grouped['std'].fillna(0),
                marker='s', capsize=4, color='#c0504d')
    ax.set_xlabel('Block (runNumber)')
    ax.set_ylabel('Mittlere normalisierte Zeit (ms pro 1000 px)')
    ax.set_title('Distanz-normalisierte Zeit pro Block - ' + label)
    ax.set_xticks(grouped['runNumber'])
    ax.grid(True, alpha=0.3)
    return save_fig(fig, out_dir, prefix + 'normalized_time_per_block.png')


def plot_flicks_per_block(df, label, out_dir, prefix):
    if not has_flick_data(df):
        return None
    valid = df.dropna(subset=['flickCount'])
    grouped = valid.groupby('runNumber')['flickCount'].agg(['mean', 'std']).reset_index()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.errorbar(grouped['runNumber'], grouped['mean'], yerr=grouped['std'].fillna(0),
                marker='o', capsize=4, color='#7b4fbb')
    ax.set_xlabel('Block (runNumber)')
    ax.set_ylabel('Mittlere Anzahl Flicks pro Versuch')
    ax.set_title('Benoetigte Flicks pro Block - ' + label)
    ax.set_xticks(grouped['runNumber'])
    ax.grid(True, alpha=0.3)
    return save_fig(fig, out_dir, prefix + 'flicks_per_block.png')


def plot_parameter_evolution(df, label, out_dir, prefix):
    params = df.groupby('runNumber')[['decelerationRate', 'scrollFriction', 'inflexion']].first().reset_index()
    specs = [('decelerationRate', '#2c6fbb'), ('scrollFriction', '#4c9a2a'), ('inflexion', '#c0504d')]
    fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
    for ax, (col, color) in zip(axes, specs):
        ax.plot(params['runNumber'], params[col], marker='o', color=color)
        ax.set_ylabel(col)
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel('Block (runNumber)')
    axes[-1].set_xticks(params['runNumber'])
    axes[0].set_title('Getestete Parameter pro Block - ' + label)
    fig.tight_layout()
    return save_fig(fig, out_dir, prefix + 'parameter_evolution.png')

In [22]:
def evaluate_file(path):
    participants = load_results_file(path)
    out_dir = plots_subdir_for(path)
    multi = len(participants) > 1
    saved = []
    for participant in participants:
        df = attempts_dataframe(participant)
        if df.empty:
            print('  Ueberspringe Teilnehmer ohne Versuche:', participant.get('participantId'))
            continue
        label = participant_display_name(participant)
        prefix = (short_id(participant) + '_') if multi else ''
        plots = [
            plot_normalized_time_per_block(df, label, out_dir, prefix),
            plot_flicks_per_block(df, label, out_dir, prefix),
            plot_parameter_evolution(df, label, out_dir, prefix),
        ]
        if not has_flick_data(df):
            print('  Hinweis: keine Flick-Daten fuer', label, '(aeltere Aufzeichnung) - Flick-Plots uebersprungen.')
        saved.extend(p for p in plots if p is not None)
    return saved


if not pending_files:
    print('Alle Ergebnisdateien haben bereits Plots. Setze FORCE_REGENERATE = True zum Neuerstellen.')
else:
    for path in pending_files:
        print('Werte aus:', path.name)
        saved = evaluate_file(path)
        for s in saved:
            print('  gespeichert:', s.relative_to(REPO_ROOT))
    print('Fertig.')

Werte aus: touch-scrolling-data-p3-Ur-Kr-2026-09-14.json
  gespeichert: plots\touch-scrolling-data-p3-Ur-Kr-2026-09-14\normalized_time_per_block.png
  gespeichert: plots\touch-scrolling-data-p3-Ur-Kr-2026-09-14\flicks_per_block.png
  gespeichert: plots\touch-scrolling-data-p3-Ur-Kr-2026-09-14\parameter_evolution.png
Fertig.


## Hinweise

- Neue Ergebnisdateien einfach in `ergebnisse/` ablegen und dieses Notebook erneut ausfuehren. Nur Dateien ohne vorhandene Plots werden verarbeitet.
- Um bestehende Plots neu zu erzeugen: `FORCE_REGENERATE = True` in der zweiten Codezelle setzen.
- Enthaelt eine Datei mehrere Teilnehmer, werden die Dateinamen mit der Kurz-ID des Teilnehmers praefixiert.